# Keystone Eval Deep Dive — `2026-03-08_four_model_thad_v2`

Load all eval results from S3 into a DataFrame (one row per trial),
then plot CDFs of duration and cost to sanity-check the data.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import s3fs

S3_PREFIX = "int8-datasets/keystone/evals"
RUN_NAME = "2026-03-08_four_model_thad_v2"

fs = s3fs.S3FileSystem(anon=False)

In [ ]:
def load_run_from_s3(run_name: str) -> list[dict]:
    """Load every eval_result.json for a run into a flat list of record dicts."""
    run_prefix = f"{S3_PREFIX}/{run_name}"
    records: list[dict] = []

    for model_path in sorted(fs.ls(run_prefix, detail=False)):
        model_name = model_path.rstrip("/").split("/")[-1]
        try:
            repo_dirs = fs.ls(model_path, detail=False)
        except FileNotFoundError:
            continue

        for repo_path in sorted(repo_dirs):
            repo_name = repo_path.rstrip("/").split("/")[-1]
            result_path = f"{repo_path}/trial_0/eval_result.json"
            if not fs.exists(result_path):
                continue
            with fs.open(result_path) as f:
                d = json.load(f)

            br = d.get("bootstrap_result") or {}
            agent = br.get("agent") or {}
            cost_info = agent.get("cost") or {}
            verification = br.get("verification") or {}
            repo_entry = d.get("repo_entry") or {}
            token_spending = cost_info.get("token_spending") or {}

            error_msg = br.get("error_message") or ""

            records.append({
                "run": run_name,
                "model": model_name,
                "repo": repo_name,
                "s3_trial_path": f"s3://{repo_path}/trial_0",
                "success": d.get("success", False),
                "bootstrap_success": br.get("success", False),
                "verification_success": verification.get("success", False),
                "error_message": error_msg,
                "error_category": categorize_error(error_msg),
                "agent_duration_s": agent.get("duration_seconds"),
                "agent_timed_out": agent.get("timed_out", False),
                "cost_usd": cost_info.get("cost_usd"),
                "input_tokens": token_spending.get("input"),
                "output_tokens": token_spending.get("output"),
                "image_build_seconds": verification.get("image_build_seconds"),
                "test_execution_seconds": verification.get("test_execution_seconds"),
                "tests_passed": verification.get("tests_passed"),
                "tests_failed": verification.get("tests_failed"),
                "language": repo_entry.get("language", ""),
                "difficulty": repo_entry.get("difficulty", ""),
            })

    return records


def categorize_error(error: str) -> str:
    """Categorize a failure error message into a named bucket."""
    if not error:
        return "(none)"
    e = error.lower()
    if "dockerfile not found" in e:
        return "No files created"
    if "timeout" in e or "timed out" in e or "status timeout" in e:
        return "Agent timeout"
    if "not found" in e and "already shut down" in e:
        return "Sandbox expired"
    if "associated container has finished" in e:
        return "Sandbox container finished"
    if "container id" in e and ("finished" in e or "status=" in e):
        return "Sandbox container crashed"
    if "no container with id" in e:
        return "Sandbox container not found"
    if "build failed" in e:
        return "Docker build failed"
    if "test run failed" in e or ("test" in e and "return code" in e):
        return "Tests failed"
    if any(x in e for x in ["nodename nor servname", "file descriptor not found", "errno", "eof"]):
        return "Infrastructure error"
    return "Other"


records = load_run_from_s3(RUN_NAME)
df = pd.DataFrame(records)
print(f"Loaded {len(df)} trials across {df['model'].nunique()} models, {df['repo'].nunique()} repos")
print()
print(df.groupby("model").agg(
    trials=("success", "count"),
    passes=("success", "sum"),
    pass_rate=("success", "mean"),
).assign(pass_rate=lambda x: (x["pass_rate"] * 100).round(1)))
df.head()

## Duration CDF by Model

Are the 0-second durations real, or data artifacts?

In [ ]:
# Quick look at the 0-duration trials
zeros = df[df["agent_duration_s"].fillna(0) == 0]
print(f"{len(zeros)} trials with duration == 0 or NaN:")
print(zeros.groupby("model")["error_category"].value_counts().to_string())
print()
print("Sample error messages from 0-duration trials:")
for _, row in zeros.head(5).iterrows():
    msg = (row["error_message"] or "")[:200]
    print(f"  [{row['model']}] {row['repo']}: {msg}")

In [ ]:
dur = df.copy()
dur["duration_min"] = dur["agent_duration_s"] / 60

fig = px.ecdf(
    dur,
    x="duration_min",
    color="model",
    title=f"Agent Duration CDF — {RUN_NAME}",
    labels={"duration_min": "Agent Duration (minutes)", "model": "Model"},
    ecdfnorm="percent",
)
fig.update_layout(xaxis_range=[0, dur["duration_min"].quantile(0.99)], yaxis_title="% of trials")
fig.show()

In [ ]:
# Same CDF but excluding 0-duration trials to see the "real" distribution
dur_nonzero = dur[dur["agent_duration_s"] > 0]

fig = px.ecdf(
    dur_nonzero,
    x="duration_min",
    color="model",
    title=f"Agent Duration CDF (excluding 0s) — {RUN_NAME}",
    labels={"duration_min": "Agent Duration (minutes)", "model": "Model"},
    ecdfnorm="percent",
)
fig.update_layout(yaxis_title="% of trials")
fig.show()

## Cost CDF by Model

Are we getting cost measurements every time the agent actually runs?

In [ ]:
# Check: which trials have duration > 0 but cost == 0 or NaN?
ran = df[df["agent_duration_s"] > 0].copy()
missing_cost = ran[(ran["cost_usd"].isna()) | (ran["cost_usd"] == 0)]

print(f"Trials where agent ran (duration > 0): {len(ran)}")
print(f"  ... but cost is 0 or NaN: {len(missing_cost)}")
print()
if len(missing_cost) > 0:
    print("Breakdown by model:")
    print(missing_cost.groupby("model")[["success", "cost_usd", "agent_duration_s"]].agg(
        count=("success", "count"),
        passes=("success", "sum"),
        mean_duration=("agent_duration_s", "mean"),
    ))
    print()
    print("Sample missing-cost trials:")
    for _, row in missing_cost.head(5).iterrows():
        print(f"  [{row['model']}] {row['repo']}: duration={row['agent_duration_s']:.0f}s, "
              f"cost={row['cost_usd']}, success={row['success']}")

In [ ]:
cost = df[df["cost_usd"].notna() & (df["cost_usd"] > 0)].copy()

fig = px.ecdf(
    cost,
    x="cost_usd",
    color="model",
    title=f"Cost CDF (trials with cost > 0) — {RUN_NAME}",
    labels={"cost_usd": "Cost (USD)", "model": "Model"},
    ecdfnorm="percent",
)
fig.update_layout(yaxis_title="% of trials")
fig.show()

In [ ]:
# Cost vs duration scatter — do they correlate? Color by success.
scatter = df[(df["agent_duration_s"] > 0) & (df["cost_usd"].notna())].copy()
scatter["duration_min"] = scatter["agent_duration_s"] / 60
scatter["outcome"] = scatter["success"].map({True: "pass", False: "fail"})

fig = px.scatter(
    scatter,
    x="duration_min",
    y="cost_usd",
    color="model",
    symbol="outcome",
    hover_data=["repo", "error_category"],
    title=f"Cost vs Duration — {RUN_NAME}",
    labels={"duration_min": "Duration (min)", "cost_usd": "Cost (USD)"},
    opacity=0.6,
)
fig.show()

## Failure Breakdown

In [ ]:
failures = df[~df["success"]].copy()

print(f"{len(failures)} failures out of {len(df)} trials ({100*len(failures)/len(df):.1f}%)")
print()
print("By model x error category:")
ct = pd.crosstab(failures["model"], failures["error_category"], margins=True)
ct

In [ ]:
fig = px.bar(
    failures.groupby(["model", "error_category"]).size().reset_index(name="count"),
    x="model",
    y="count",
    color="error_category",
    title=f"Failure Categories by Model — {RUN_NAME}",
    labels={"count": "Failures", "error_category": "Category"},
)
fig.show()